In [ ]:
import os, warnings
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from src.plotting import plot_autocorrelation_analysis, plot_subject_timeseries_heatmap

warnings.simplefilter("ignore", UserWarning)

In [ ]:
dataset = 'HCP-YA'
# dataset = 'WAND'
# dataset = 'HCP-D'
# dataset = 'PNC'
# dataset = 'Penn_LEAD'
# dataset = 'UCLA'
# dataset = 'NKI'
# dataset = 'HBN'

data_loader = DataLoader(dataset=dataset)

# Load fMRI data

In [ ]:
seg = 'Schaefer4007'
use_subsample = False
run_mri3t = True
run_mri7t = False
run_meg = False

if dataset == 'HCP-YA':
    run_mri3t = True
    run_mri7t = True
    # run_meg = True
    nuisance_regression = False
    task_regression = True

    if use_subsample:
        subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_subsample.txt'), dtype=str)
    else:
        subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)

    sessions = ['01',]
    tasks = ['rfMRIREST1LR', 'rfMRIREST1RL', 'rfMRIREST2LR', 'rfMRIREST2RL',
             'tfMRIWMLR', 'tfMRIWMRL',
             'tfMRIEMOTIONLR', 'tfMRIEMOTIONRL',
             'tfMRIGAMBLINGLR', 'tfMRIGAMBLINGRL',
             'tfMRILANGUAGELR', 'tfMRILANGUAGERL',
             'tfMRIRELATIONALLR', 'tfMRIRELATIONALRL',
             'tfMRISOCIALLR', 'tfMRISOCIALRL']
            #  'tfMRIMOTORLR', 'tfMRIMOTORRL']
    # tasks = ['tfMRIWMLR', 'tfMRIWMRL']
    tasks_7t = ['rfMRIREST17TPA', 'rfMRIREST27TAP', 'rfMRIREST37TPA', 'rfMRIREST47TAP',
                'tfMRIMOVIE17TAP', 'tfMRIMOVIE27TPA', 'tfMRIMOVIE37TPA', 'tfMRIMOVIE47TAP']
    tasks_meg = ['3Restin', '4Restin', '5Restin', '6Wrkmem', '7Wrkmem']
    space = 'fsLR'
elif dataset == 'HCP-D':
    nuisance_regression = False
    # nuisance_regression = True
    task_regression = True

    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    # subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_subset.txt'), dtype=str)
    subject_ids = [subj_id.replace('_', '') for subj_id in subject_ids]
    sessions = ['01',]
    tasks = ['rfMRIREST1PA', 'rfMRIREST2PA',
             'tfMRICARITPA', 'tfMRIEMOTIONPA', 'tfMRIGUESSINGPA']
    space = 'fsLR'
elif dataset == 'WAND':
    nuisance_regression = False
    task_regression = False

    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_bids.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['03',]
    tasks = ['rest', 'reversallearning']
    space = 'MNI152NLin2009cAsym'
elif dataset == 'PNC':
    nuisance_regression = False
    task_regression = True

    seg = 'Schaefer2018p400n17'
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_lowmotion.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['PNC1',]
    tasks = ['rest', 'frac2back']
    space = 'MNI152NLin6ASym'
elif dataset == 'UCLA':
    nuisance_regression = False
    task_regression = True

    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_not1_hc.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = [None,]
    tasks = ['rest', 'bart', 'stopsignal'] # 'taskswitch']
    space = 'MNI152NLin2009cAsym'
elif dataset == 'NKI':
    nuisance_regression = False
    task_regression = True
    
    seg = 'Schaefer2018p400n17'
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['BAS1',]
    tasks = ['rest', 'CHECKERBOARD']  # 'BREATHHOLD'
    space = 'MNI152NLin6ASym'
elif dataset == 'Penn_LEAD':
    nuisance_regression = False
    task_regression = True

    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['1',]
    tasks = ['rest', 'nback']
    space = 'MNI152NLin2009cAsym'
elif dataset == 'HBN':
    nuisance_regression = False
    task_regression = False
    
    seg = 'Schaefer2018p400n17'
    # subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_HBNsiteCBIC.txt'), dtype=str)
    # subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_HBNsiteRU.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    # sessions = ['HBNsiteCBIC', 'HBNsiteRU']
    sessions = ['HBNsiteCBIC',]
    # sessions = ['HBNsiteRU',]
    tasks = ['rest',]
    space = 'MNI152NLin6ASym'

# subject_ids = subject_ids[:20]
# subject_ids = subject_ids[:int(len(subject_ids)/2)]
# subject_ids = subject_ids[int(len(subject_ids)/2):]
subject_ids

In [ ]:
if run_mri3t:
    scans = data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=sessions,
        tasks=tasks,
        space=space,
        seg=seg,
        modality='MRI3T'
    )

if run_mri7t:
    scans_7t = data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=sessions,
        tasks=tasks_7t,
        space=space,
        seg=seg,
        modality='MRI7T'
    )
    
if run_meg:
    scans = data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=sessions,
        tasks=tasks_meg,
        space=space,
        seg=seg,
        modality='MEG'
    )

In [ ]:
_ = data_loader.validate_scan_registry(remove_incomplete_subjects=True)

In [ ]:
# Load the data
timeseries_data = data_loader.load_batch(
    nuisance_regression=nuisance_regression,
    standardize=False,
    verbose=True
)

print(f"Loaded {len(timeseries_data)} scans")
print(timeseries_data.keys())

In [ ]:
if use_subsample:
    data_loader.export_scan_registry(f'scan_registry_{dataset}_subsample.csv')
else:
    if dataset == 'HCP-YA' and run_mri7t:
        data_loader.export_scan_registry(f'scan_registry_{dataset}_7T.csv')
    elif dataset == 'HCP-YA' and run_meg:
        data_loader.export_scan_registry(f'scan_registry_{dataset}_MEG.csv')
    else:
        data_loader.export_scan_registry(f'scan_registry_{dataset}.csv')
scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['sub'].unique()
n_subs = len(subject_ids)
print(n_subs)

In [ ]:
subject_ids

## Compute functional connectivity

In [ ]:
# def compute_fc(ts):
#     """
#     Parameters
#     ----------
#     ts : np.array (n_parcels, n_timepoints)
#         time series

#     Returns
#     -------
#     fc : np.array (n_parcels, n_parcels)
#         functional connectivity matrix
#     """

#     fc = np.corrcoef(ts, rowvar=True)
#     np.fill_diagonal(fc, np.nan)
#     fc = np.arctanh(fc)
#     np.fill_diagonal(fc, 1)

#     return fc

In [ ]:
# from tqdm import tqdm
# import json

In [ ]:
# functional_connectivity = dict().fromkeys(timeseries_data)

# for idx in tqdm(scan_registry.index):
#     scan_string = scan_registry.loc[idx, 'scan_string']

#     raw_data = timeseries_data[scan_string]
#     functional_connectivity[scan_string] = compute_fc(raw_data)

In [ ]:
# if dataset == 'HCP-YA':
#     np.savez('/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/HCP-YA_fc', **functional_connectivity)

# Process fMRI timescales

In [ ]:
timescales_processor = Timescales(dataset=dataset, timeseries_data=timeseries_data, verbose=True)
if task_regression:
    timescales_processor.regress_task_structure()

### Save out time series for Emily

In [ ]:
run_cell = False

In [ ]:
if run_cell:
    timeseries_data_out = dict()
    for scan_key in timescales_processor.timeseries_data.keys():
        if 'task-rfMRIREST' in scan_key:
            timeseries_data_out[scan_key] = np.array(timescales_processor.timeseries_data[scan_key]).copy()
    print(len(list(timeseries_data_out.keys())))
    np.save(os.path.expanduser('~/Desktop/HCP-YA_Schaefer400-7_task-rest_data-ts.npy'), timeseries_data_out)

    timeseries_data_out = dict()
    for scan_key in timescales_processor.timeseries_data.keys():
        if 'task-tfMRIWM' in scan_key:
            timeseries_data_out[scan_key] = np.array(timescales_processor.timeseries_data[scan_key]).copy()
    print(len(list(timeseries_data_out.keys())))      
    np.save(os.path.expanduser('~/Desktop/HCP-YA_Schaefer400-7_task-wm_data-ts.npy'), timeseries_data_out)

    timeseries_data_out = dict()
    for scan_key in timescales_processor.timeseries_residuals.keys():
        if 'task-tfMRIWM' in scan_key:
            timeseries_data_out[scan_key] = np.array(timescales_processor.timeseries_residuals[scan_key]).copy()
    print(len(list(timeseries_data_out.keys())))  
    np.save(os.path.expanduser('~/Desktop/HCP-YA_Schaefer400-7_task-wm_data-tsresid.npy'), timeseries_data_out)

In [ ]:
timescales_processor.get_min_scan_length_per_subject()

### Timeseries metrics (not timescales)

In [ ]:
run_cell = True

In [ ]:
if run_cell:
    for method in ['tsnr', 'var']:
        timescales_processor.compute_timeseries_measures(method=method, force_recompute=False, verbose=True)

        if task_regression:
            timescales_processor.compute_timeseries_measures(method=method, use_residuals=True, force_recompute=False, verbose=True)

### Timescale measures

In [ ]:
run_cell = True

In [ ]:
timescale_measures = [
    'zero_crossing',
    'zero_crossing_robust_nc2',
    'zero_crossing_robust_nc3',
    'auc_zero_crossing',
    # 'auc_half_crossing',
    # 'decay_time_half',
    # 'lag_one',

    # 'tau',
    # 'tau_howell',
    # 'tau_murray_par',
    # 'B_murray_par',
    # 'B_murray_par_lag3mins',
    # 'B_murray_par_lag4mins',
    # 'B_murray_par_lag5mins',
]

In [ ]:
if run_cell:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, force_recompute=False, verbose=True)
        
    autocorr_dict_raw = timescales_processor.autocorr_dict.copy()
    timescales_dict_raw = timescales_processor.timescales_dict.copy()

In [ ]:
if run_cell and task_regression:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix='resids', force_recompute=False, verbose=True)
        
    autocorr_dict_resid = timescales_processor.autocorr_dict.copy()
    timescales_dict_resid = timescales_processor.timescales_dict.copy()

### Frequency specific timescales (MEG only)

In [ ]:
if dataset == 'HCP-YA' and run_meg and not run_mri3t and not run_mri7t:
    timescale_measures = [
        'zero_crossing',
    ]

    # lowpass_cutoffs = [50, 30, 10, 5, 1, 0.5, 0.1]
    lowpass_cutoffs = [1, 0.5, 0.1]

    for lowpass_cutoff in lowpass_cutoffs:
        timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False, lowpass_cutoff=lowpass_cutoff, verbose=True)
        for timescale_measure in timescale_measures:
            timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, suffix=f'lp-{lowpass_cutoff}hz', force_recompute=False, verbose=True)

    autocorr_dict_raw = timescales_processor.autocorr_dict
    timescales_dict_raw = timescales_processor.timescales_dict     
    autocorr_dict_resid = timescales_processor.autocorr_dict
    timescales_dict_resid = timescales_processor.timescales_dict
    
    if task_regression:
        for lowpass_cutoff in lowpass_cutoffs:
            timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False, lowpass_cutoff=lowpass_cutoff, verbose=True)
            for timescale_measure in timescale_measures:
                timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix=f'resids_lp-{lowpass_cutoff}hz', force_recompute=False, verbose=True)

### Condition analysis (fMRI and WM task only)

In [ ]:
def extract_wm_block_indices(design_matrix, TR):
    """
    Extract integer indices for 0-back and 2-back task blocks from design matrix.
    Returns only exclusive timepoints (no overlap between tasks).
    
    Parameters
    ----------
    design_matrix : pd.DataFrame
        Design matrix dataframe with time as index and task conditions as columns
    TR : float
        Repetition time in seconds (e.g., 0.72)
    
    Returns
    -------
    idx_0bk : ndarray of int
        Integer indices of exclusive 0-back timepoints (concatenated across conditions)
    idx_2bk : ndarray of int
        Integer indices of exclusive 2-back timepoints (concatenated across conditions)
    """
    # Get time column (first column after index)
    # Actually, the index IS the time column based on the CSV structure
    time = design_matrix.index.values
    
    # Identify 0bk columns
    obk_cols = [col for col in design_matrix.columns if col.startswith('0bk_')]
    
    # Identify 2bk columns  
    tbk_cols = [col for col in design_matrix.columns if col.startswith('2bk_')]
    
    # Find timepoints where any 0bk condition is active (non-zero)
    obk_active = (design_matrix[obk_cols] > 0).any(axis=1)
    
    # Find timepoints where any 2bk condition is active (non-zero)
    tbk_active = (design_matrix[tbk_cols] > 0).any(axis=1)
    
    # Find exclusive timepoints (only one task active)
    obk_exclusive = obk_active & ~tbk_active
    tbk_exclusive = tbk_active & ~obk_active
    
    # Get time values where tasks are exclusively active
    time_0bk = time[obk_exclusive]
    time_2bk = time[tbk_exclusive]
    
    # Convert time to TR indices
    idx_0bk = np.round(time_0bk / TR).astype(int)
    idx_2bk = np.round(time_2bk / TR).astype(int)
    
    return idx_0bk, idx_2bk

In [ ]:
run_cell = False

In [ ]:
if run_cell:
    timeseries_data_0bk = dict.fromkeys(timescales_processor.timeseries_data.keys())
    timeseries_residuals_0bk = dict.fromkeys(timescales_processor.timeseries_residuals.keys())
    timeseries_data_2bk = dict.fromkeys(timescales_processor.timeseries_data.keys())
    timeseries_residuals_2bk = dict.fromkeys(timescales_processor.timeseries_residuals.keys())
    
    for idx in scan_registry.index:
        scan_string = scan_registry.loc[idx, 'scan_string']
        tr = scan_registry.loc[idx, 'tr']
        design_matrix = timescales_processor.design_matrices[scan_string]
        idx_0bk, idx_2bk = extract_wm_block_indices(design_matrix, TR=tr)

        timeseries_data_0bk[scan_string] = timescales_processor.timeseries_data[scan_string][:, idx_0bk]
        timeseries_residuals_0bk[scan_string] = timescales_processor.timeseries_residuals[scan_string][:, idx_0bk]
        timeseries_data_2bk[scan_string] = timescales_processor.timeseries_data[scan_string][:, idx_2bk]
        timeseries_residuals_2bk[scan_string] = timescales_processor.timeseries_residuals[scan_string][:, idx_2bk]
    

In [ ]:
if run_cell:
    # replace time series data with filtered data
    timescales_processor.timeseries_data = timeseries_data_0bk
    timescales_processor.timeseries_residuals = timeseries_residuals_0bk
    # recompute metrics
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, suffix='0bk', force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, suffix='0bk', force_recompute=False, verbose=True)
    if task_regression:
        timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, suffix='0bk', force_recompute=False, verbose=True)
        for timescale_measure in timescale_measures:
            timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix='0bk_resids', force_recompute=False, verbose=True)

        
    # replace time series data with filtered data
    timescales_processor.timeseries_data = timeseries_data_2bk
    timescales_processor.timeseries_residuals = timeseries_residuals_2bk
    # recompute metrics
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, suffix='2bk', force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, suffix='2bk', force_recompute=False, verbose=True)
    if task_regression:
        timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, suffix='2bk', force_recompute=False, verbose=True)
        for timescale_measure in timescale_measures:
            timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix='2bk_resids', force_recompute=False, verbose=True)

# Generate plots

In [ ]:
run_cell = False

In [ ]:
from src.timescales import apply_lowpass_filter

In [ ]:
if run_cell:
    for idx in scan_registry.index[:25]:
        # if idx & 10 == 0 and scan_registry.loc[idx, 'condition'] == 'task':
        if scan_registry.loc[idx, 'condition'] == 'task':
            fig_height = 10
        else:
            fig_height = 3
        scan_string = scan_registry.loc[idx, 'scan_string']
        tr = scan_registry.loc[idx, 'tr']
        print(idx, scan_string)
    
        raw_data = timescales_processor.timeseries_data[scan_string]
        # lowpass_cutoff = 5
        # raw_data = apply_lowpass_filter(raw_data, sampling_freq=1/tr, lowpass_cutoff=lowpass_cutoff)
        try:
            residual_data = timescales_processor.timeseries_residuals[scan_string]
            # residual_data = apply_lowpass_filter(residual_data, sampling_freq=1/tr, lowpass_cutoff=lowpass_cutoff)
            design_matrix = timescales_processor.design_matrices[scan_string]
        except:
            residual_data = None
            design_matrix = None

        plot_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'subject_plots', 'plot_subject_timeseries_heatmap')
        if not os.path.exists(plot_dir):
            os.makedirs(plot_dir)

        fig, axes = plot_subject_timeseries_heatmap(raw_data, scan_string,
                                                    residual_data, design_matrix,
                                                    figsize=(15, fig_height), dpi=100,
                                                    save_path=os.path.join(plot_dir, f'{scan_string}.png')
                                                    )
        plt.close('all')

In [ ]:
run_cell = True

In [ ]:
if run_cell:
    for idx in scan_registry.index[:20]:
    # for idx in [2]:
        # if idx & 10 == 0 and scan_registry.loc[idx, 'condition'] == 'task':
        if scan_registry.loc[idx, 'condition'] == 'task':
            scan_string = scan_registry.loc[idx, 'scan_string']
            tr = scan_registry.loc[idx, 'tr']
            print(idx, scan_string)
        
            autocorr_raw = autocorr_dict_raw[scan_string]
            autocorr_resid = autocorr_dict_resid[scan_string]
            timescales_raw = timescales_dict_raw[scan_string]
            timescales_resid = timescales_dict_resid[scan_string]

            plot_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'subject_plots', 'plot_autocorrelation_analysis')
            if not os.path.exists(plot_dir):
                os.makedirs(plot_dir)

            fig = plot_autocorrelation_analysis(autocorr_raw, autocorr_resid, timescales_raw, timescales_resid, 
                                                tr, figsize=(15, 4), dpi=100, max_lag=50,
                                                save_path=os.path.join(plot_dir, f'{scan_string}.png')
                                                )
            plt.close('all')